## Đây là code phần training và test Llama2 trên benchmark Stanford Sentiment Treebank (sst dataset) được sinh ra từ prompt trên Claude. Chỉ coi đây là code tham khảo, không phải code đúng 100%

## Kiểm tra lại và sửa để train và test được (nếu cần)

## Ngoài sst dataset, train và test với cả cfimdb dataset (yêu cầu bắt buộc của assignment)

In [1]:
# Cell 1: Imports & Setup
import os
import math
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score, classification_report

import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cpu


In [3]:
# Cell 2: Hyperparameters & Model Config
from llama2_components import ModelArgs   # file của bạn
from llama2 import Llama2Model      # file của bạn

# ── Dùng model nhỏ hơn cho classification ──────────────────────────────────
MODEL_CFG = dict(
    dim          = 512,
    n_layers     = 6,
    n_heads      = 8,
    n_kv_heads   = 4,
    vocab_size   = -1,          # sẽ điền sau khi load tokenizer
    multiple_of  = 32,
    norm_eps     = 1e-5,
    rope_base    = 10000.0,
    max_batch_size = 32,
    max_seq_len    = 128,
    device         = DEVICE,
)

TRAIN_CFG = dict(
    batch_size    = 32,
    lr            = 3e-4,
    weight_decay  = 1e-2,
    epochs        = 10,
    max_seq_len   = 128,
    num_classes   = 2,          # SST-2: positive / negative
    warmup_steps  = 200,
)

In [ ]:
# Cell 3: Tokenizer & SST-2 Dataset
TOKENIZER_NAME = "hf-internal-testing/llama-tokenizer"  # Llama tokenizer, không cần token

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

MODEL_CFG["vocab_size"] = tokenizer.vocab_size
print(f"Vocab size: {tokenizer.vocab_size}")  # → 32000

In [ ]:
# Cell 3b: Load SST-2
raw = load_dataset("glue", "sst2")
print(raw)

class SSTDataset(Dataset):
    """Wrapper Dataset cho SST-2."""
    def __init__(self, hf_split, tokenizer, max_len: int):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc = self.tokenizer(
            item["sentence"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)   # (max_len,)
        label     = torch.tensor(item["label"], dtype=torch.long)
        return input_ids, label

max_len = TRAIN_CFG["max_seq_len"]
train_ds = SSTDataset(raw["train"],      tokenizer, max_len)
val_ds   = SSTDataset(raw["validation"], tokenizer, max_len)

train_loader = DataLoader(train_ds, batch_size=TRAIN_CFG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=TRAIN_CFG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
# Cell 4: Llama2 Classifier

class Llama2ForClassification(nn.Module):
    """
    Wrap Llama2Model thêm classification head.
    Strategy: lấy hidden state tại vị trí token cuối cùng (last non-pad token)
    rồi project sang num_classes.
    """
    def __init__(self, args: ModelArgs, num_classes: int):
        super().__init__()
        self.backbone = Llama2Model(args)
        # Xóa output projection của LM head (không cần thiết)
        self.backbone.output_proj = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Linear(args.dim, args.dim // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(args.dim // 2, num_classes),
        )
        self.pad_token_id = tokenizer.pad_token_id

    def reset_kv_cache(self):
        for block in self.backbone.blocks:
            block.attn.reset_cache()

    def _last_token_hidden(self, hidden: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
        """Lấy hidden state của token cuối không phải padding."""
        # Tìm vị trí token cuối thực sự (không phải pad)
        mask = (input_ids != self.pad_token_id).long()          # (B, seq_len)
        last_pos = mask.sum(dim=1) - 1                           # (B,)
        last_pos = last_pos.clamp(min=0)
        # Gather: (B, dim)
        idx = last_pos.unsqueeze(-1).unsqueeze(-1).expand(-1, 1, hidden.size(-1))
        return hidden.gather(1, idx).squeeze(1)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # Llama2Model.forward nhận (tokens, start_pos)
        # start_pos=0 vì chúng ta train theo full sequence (không decode auto-regressive)
        hidden = self.backbone(input_ids, start_pos=0)  # (B, seq_len, dim)
        pooled = self._last_token_hidden(hidden, input_ids)  # (B, dim)
        return self.classifier(pooled)                        # (B, num_classes)

In [ ]:
# Cell 4b: Khởi tạo model
args = ModelArgs(**MODEL_CFG)
model = Llama2ForClassification(args, num_classes=TRAIN_CFG["num_classes"]).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params:,}")

In [ ]:
# Cell 5: Optimizer & Scheduler

optimizer = AdamW(
    model.parameters(),
    lr=TRAIN_CFG["lr"],
    weight_decay=TRAIN_CFG["weight_decay"],
    betas=(0.9, 0.95),
)

total_steps   = len(train_loader) * TRAIN_CFG["epochs"]
warmup_steps  = TRAIN_CFG["warmup_steps"]

def lr_lambda(step: int) -> float:
    """Linear warmup + cosine decay."""
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Cell 6: Train / Eval helpers

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []

    for input_ids, labels in tqdm(loader, desc="Train", leave=False):
        input_ids = input_ids.to(device)
        labels    = labels.to(device)
        model.reset_kv_cache()
        optimizer.zero_grad()
        logits = model(input_ids)                   # (B, num_classes)
        loss   = criterion(logits, labels)
        loss.backward()

        # Gradient clipping để ổn định training
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for input_ids, labels in tqdm(loader, desc="Eval ", leave=False):
        input_ids = input_ids.to(device)
        labels    = labels.to(device)

        logits = model(input_ids)
        loss   = criterion(logits, labels)

        total_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1, all_preds, all_labels

In [ ]:
# Cell 7: Training loop

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc  = 0.0
best_ckpt     = "best_llama2_sst2.pt"

for epoch in range(1, TRAIN_CFG["epochs"] + 1):
    print(f"\n{'='*55}")
    print(f"  Epoch {epoch}/{TRAIN_CFG['epochs']}  |  lr = {scheduler.get_last_lr()[0]:.2e}")
    print(f"{'='*55}")

    tr_loss, tr_acc, tr_f1 = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, DEVICE
    )
    va_loss, va_acc, va_f1, _, _ = evaluate(
        model, val_loader, criterion, DEVICE
    )

    print(f"  Train  loss={tr_loss:.4f}  acc={tr_acc:.4f}  f1={tr_f1:.4f}")
    print(f"  Val    loss={va_loss:.4f}  acc={va_acc:.4f}  f1={va_f1:.4f}")

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)

    # Lưu checkpoint tốt nhất
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_acc": va_acc,
        }, best_ckpt)
        print(f"  ✅ Saved best checkpoint (val_acc={va_acc:.4f})")

print(f"\nBest val accuracy: {best_val_acc:.4f}")

In [ ]:
# Cell 8: Visualize

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs, history["train_loss"], label="Train", marker="o")
axes[0].plot(epochs, history["val_loss"],   label="Val",   marker="s")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(epochs, history["train_acc"], label="Train", marker="o")
axes[1].plot(epochs, history["val_acc"],   label="Val",   marker="s")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

In [ ]:
# Cell 9: Load best checkpoint & full evaluation

ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (val_acc={ckpt['val_acc']:.4f})")

_, _, _, val_preds, val_labels = evaluate(model, val_loader, criterion, DEVICE)

label_names = ["Negative", "Positive"]
print("\nClassification Report (Validation set):")
print(classification_report(val_labels, val_preds, target_names=label_names))

In [ ]:
# Cell 10: Inference on custom sentences

@torch.no_grad()
def predict(sentences: list[str], model, tokenizer, device, max_len: int = 128) -> list[str]:
    model.eval()
    enc = tokenizer(
        sentences,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids = enc["input_ids"].to(device)
    logits = model(input_ids)
    probs  = F.softmax(logits, dim=-1).cpu()
    preds  = probs.argmax(dim=-1).tolist()

    results = []
    label_map = {0: "Negative 😞", 1: "Positive 😊"}
    for sent, pred, prob in zip(sentences, preds, probs):
        confidence = prob[pred].item()
        results.append({
            "sentence":   sent,
            "label":      label_map[pred],
            "confidence": f"{confidence:.2%}",
        })
    return results

# ── Test ──────────────────────────────────────────────────────────────────
test_sentences = [
    "This movie was absolutely fantastic and I loved every minute of it!",
    "Terrible film, complete waste of time.",
    "It was okay, nothing special.",
]

for r in predict(test_sentences, model, tokenizer, DEVICE):
    print(f"[{r['label']}] ({r['confidence']})  →  \"{r['sentence']}\"")